## Netflix Churn: Calculated Fields
This notebook creates new columns from the cleaned dataset to support the descriptive analysis.

### Import libraries

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np

### Load the cleaned dataset
Extremely high values in `watch_hours` and `avg_watch_time_per_day` have already been brought down to a maximum limit, so a few unusual customers don't distort the results.

In [2]:
dataset=pd.read_csv("Netflix_cleaned.csv")

In [3]:
dataset

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73000,29,Africa,TV,8.99,1,Gift Card,1,29,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70000,19,Europe,Mobile,13.99,1,Gift Card,5,2,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32000,10,Asia,TV,13.99,0,Crypto,2,89,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51000,12,Oceania,TV,17.99,1,Crypto,2,21,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89000,13,Africa,Mobile,13.99,1,Crypto,2,8,Action
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,44f3ba44-b95d-4e50-a786-bac4d06f4a43,19,Female,Basic,32.77625,11,Europe,Desktop,8.99,0,Credit Card,4,92,Drama
4996,18779bcb-ba2b-41da-b751-e70b812061ec,67,Female,Basic,9.24000,2,North America,Desktop,8.99,0,PayPal,3,92,Documentary
4997,3f32e8c5-615b-4a3b-a864-db2688f7834f,66,Male,Standard,16.55000,49,South America,Desktop,13.99,1,Debit Card,2,20,Action
4998,7b0ad82d-6571-430e-90f4-906259e0e89c,59,Female,Basic,9.12000,3,Europe,Laptop,8.99,0,Credit Card,4,92,Sci-Fi


### Check for missing values

In [4]:
dataset.isna().sum()

customer_id               0
age                       0
gender                    0
subscription_type         0
watch_hours               0
last_login_days           0
region                    0
device                    0
monthly_fee               0
churned                   0
payment_method            0
number_of_profiles        0
avg_watch_time_per_day    0
favorite_genre            0
dtype: int64

In [5]:
dataset.loc[(dataset["churned"]==0,"churned")]

2       0
5       0
6       0
8       0
10      0
       ..
4991    0
4994    0
4995    0
4996    0
4998    0
Name: churned, Length: 2485, dtype: int64

### 1. churn_status
**Purpose:** readable labels for churn in charts and tables.

**Formula:** churned = 0 → "Stayed", churned = 1 → "Left"

In [6]:
dataset.loc[dataset["churned"] == 0, "churn_status"] = "Stayed"
dataset.loc[dataset["churned"] == 1, "churn_status"] = "Left"

In [7]:
dataset["churn_status"]

0         Left
1         Left
2       Stayed
3         Left
4         Left
         ...  
4995    Stayed
4996    Stayed
4997      Left
4998    Stayed
4999      Left
Name: churn_status, Length: 5000, dtype: object

### 2. revenue_lost
**Purpose:** monthly revenue lost from customers who churned.

**Formula:** churned = 1 → monthly_fee, churned = 0 → 0

In [8]:
dataset.loc[dataset["churned"] == 0, "revenue_lost"] = 0
dataset.loc[dataset["churned"] == 1, "revenue_lost"] = dataset["monthly_fee"]

In [9]:
dataset["revenue_lost"]

0        8.99
1       13.99
2        0.00
3       17.99
4       13.99
        ...  
4995     0.00
4996     0.00
4997    13.99
4998     0.00
4999     8.99
Name: revenue_lost, Length: 5000, dtype: float64

### 3. annual_revenue_lost
**Purpose:** yearly revenue lost per churned customer.

**Formula:** revenue_lost × 12

In [10]:
dataset["annual_revenue_lost"]=dataset["revenue_lost"]*12

In [11]:
dataset["annual_revenue_lost"]

0       107.88
1       167.88
2         0.00
3       215.88
4       167.88
         ...  
4995      0.00
4996      0.00
4997    167.88
4998      0.00
4999    107.88
Name: annual_revenue_lost, Length: 5000, dtype: float64

### 4. account_type
**Purpose:** separates single-user accounts from shared accounts.

**Formula:** number_of_profiles = 1 → "Single", number_of_profiles > 1 → "Shared"

In [12]:
dataset.loc[dataset["number_of_profiles"] == 1, "account_type"] = "Single"
dataset.loc[dataset["number_of_profiles"] >1, "account_type"] = "Shared"

In [13]:
dataset["number_of_profiles"].value_counts()

number_of_profiles
5    1034
2    1001
4     999
3     994
1     972
Name: count, dtype: int64

### 5. profile_group
**Purpose:** tests whether larger shared accounts stay longer.

**Formula:** number_of_profiles ≤ 3 → "1-3 profiles", number_of_profiles > 3 → "4-5 profiles"

In [14]:
dataset.loc[dataset["number_of_profiles"] <=3, "profile_group"] = "1-3 profiles"
dataset.loc[dataset["number_of_profiles"] >3, "profile_group"] = "4-5 profiles"

In [15]:
dataset["profile_group"]

0       1-3 profiles
1       4-5 profiles
2       1-3 profiles
3       1-3 profiles
4       1-3 profiles
            ...     
4995    4-5 profiles
4996    1-3 profiles
4997    1-3 profiles
4998    4-5 profiles
4999    1-3 profiles
Name: profile_group, Length: 5000, dtype: object

### 6. payment_type
**Purpose:** compares payments that renew automatically with payments made manually.

**Formula:** Credit Card, Debit Card, PayPal → "Automatic"; Gift Card, Crypto → "Manual"

*Assumption: Gift Card and Crypto are treated as manual because they typically do not auto-renew.*

In [16]:
dataset.loc[dataset["payment_method"].isin(["Credit Card", "Debit Card","PayPal"]), "payment_type"] = "Automatic"
dataset.loc[dataset["payment_method"].isin(["Gift Card" ,"Crypto"]), "payment_type"] = "Manual"

In [17]:
dataset["payment_type"]

0          Manual
1          Manual
2          Manual
3          Manual
4          Manual
          ...    
4995    Automatic
4996    Automatic
4997    Automatic
4998    Automatic
4999       Manual
Name: payment_type, Length: 5000, dtype: object

### 7. inactivity_group
**Purpose:** groups customers by days since last login to find where churn rises.

**Formula:** last_login_days → 0-7, 8-14, 15-30, 31-45, 46-60 days

In [18]:
dataset.loc[(dataset["last_login_days"] >=0 )&(dataset["last_login_days"]<=7), "inactivity_group"] = "0-7 days"
dataset.loc[(dataset["last_login_days"] >=8 )&(dataset["last_login_days"]<=14), "inactivity_group"] = "8-14 days"
dataset.loc[(dataset["last_login_days"] >=15 )&(dataset["last_login_days"]<=30), "inactivity_group"] = "15-30 days"
dataset.loc[(dataset["last_login_days"] >=31 )&(dataset["last_login_days"]<=45), "inactivity_group"] = "31-45 days"
dataset.loc[(dataset["last_login_days"] >=46 )&(dataset["last_login_days"]<=60), "inactivity_group"] = "46-60 days"

In [19]:
dataset["inactivity_group"]

0       15-30 days
1       15-30 days
2        8-14 days
3        8-14 days
4        8-14 days
           ...    
4995     8-14 days
4996      0-7 days
4997    46-60 days
4998      0-7 days
4999    15-30 days
Name: inactivity_group, Length: 5000, dtype: object

### 8. inactive_over_30_days
**Purpose:** flags customers who have not logged in for more than 30 days.

**Formula:** last_login_days > 30 → "Yes", otherwise → "No"

In [20]:
dataset.loc[(dataset["last_login_days"] >30 ),"inactive_over_30_days"] = "Yes"
dataset.loc[(dataset["last_login_days"] <=30 ),"inactive_over_30_days"] = "No"

In [21]:
dataset["inactive_over_30_days"].value_counts()

inactive_over_30_days
No     2647
Yes    2353
Name: count, dtype: int64

### 9. age_group
**Purpose:** groups customers into age bands for comparison.

**Formula:** age → 18-24, 25-34, 35-44, 45-54, 55-64, 65-70

In [22]:
dataset.loc[(dataset["age"] >=18 )&(dataset["age"]<=24), "age_group"] = "18–24"
dataset.loc[(dataset["age"] >=25 )&(dataset["age"]<=34), "age_group"] = "25–34"
dataset.loc[(dataset["age"] >=35 )&(dataset["age"]<=44), "age_group"] = "35–44"
dataset.loc[(dataset["age"] >=45 )&(dataset["age"]<=54), "age_group"] = "45–54"
dataset.loc[(dataset["age"] >=55 )&(dataset["age"]<=64), "age_group"] = "55–64"
dataset.loc[(dataset["age"] >=65)&(dataset["age"]<=70), "age_group"] = "65-70"



In [23]:
dataset["age_group"].value_counts()

age_group
35–44    1056
55–64     947
25–34     897
45–54     875
18–24     685
65-70     540
Name: count, dtype: int64

### 10. fee_per_profile
**Purpose:** how much each person on the account effectively pays (value for money).

**Formula:** monthly_fee ÷ number_of_profiles, rounded to 2 decimals

In [24]:
dataset["fee_per_profile"]=(dataset["monthly_fee"]/dataset["number_of_profiles"]).round(2)

In [25]:
dataset["fee_per_profile"]

0       8.99
1       2.80
2       7.00
3       8.99
4       7.00
        ... 
4995    2.25
4996    3.00
4997    7.00
4998    2.25
4999    4.50
Name: fee_per_profile, Length: 5000, dtype: float64

### 11. avg_watch_recalculated
**Purpose:** a consistent daily watch time for every customer, replacing `avg_watch_time_per_day`, which had median-filled values that did not match the other columns.

**Formula:** watch_hours ÷ (last_login_days + 1) × 60

If the answer is more than 1,440 minutes, it is changed to 1,440, because nobody can watch more than 24 hours (1,440 minutes) a day.

*The +1 avoids dividing by zero for customers who logged in today.*

In [26]:
dataset["avg_watch_recalculated"] = (dataset["watch_hours"] / (dataset["last_login_days"] + 1)) * 60

In [27]:
dataset["avg_watch_recalculated"]>=1440

0       False
1       False
2       False
3       False
4       False
        ...  
4995    False
4996    False
4997    False
4998    False
4999    False
Name: avg_watch_recalculated, Length: 5000, dtype: bool

In [28]:
dataset.loc[dataset["avg_watch_recalculated"] > 1440, "avg_watch_recalculated"] = 1440

### 12. viewing_level
**Purpose:** groups customers by how much they watch.

**Formula:** watch_hours < Q1 → "Low" (bottom 25%); Q1 to Q3 → "Medium" (middle 50%); > Q3 → "High" (top 25%)

In [29]:
# Quartiles of watch_hours
# Q1 = 25th percentile: 25% of customers watch less than this
# Q3 = 75th percentile: 75% of customers watch less than this
q1 = dataset["watch_hours"].quantile(0.25)
q3 = dataset["watch_hours"].quantile(0.75)

# Low: bottom 25% of viewers (below Q1)
dataset.loc[dataset["watch_hours"] < q1, "viewing_level"] = "Low"

# Medium: middle 50% of viewers (between Q1 and Q3)
# Q2 (median) is not used, because Medium covers the whole middle half
dataset.loc[(dataset["watch_hours"] >= q1) & (dataset["watch_hours"] <= q3), "viewing_level"] = "Medium"

# High: top 25% of viewers (above Q3)
dataset.loc[dataset["watch_hours"] > q3, "viewing_level"] = "High"

In [30]:
dataset["viewing_level"]

0       Medium
1          Low
2         High
3       Medium
4          Low
         ...  
4995      High
4996    Medium
4997      High
4998    Medium
4999       Low
Name: viewing_level, Length: 5000, dtype: object

### 13. high_risk_flag
**Purpose:** identifies customers showing both warning signs at once.

**Formula:** viewing_level = "Low" AND inactive_over_30_days = "Yes" → "Yes", otherwise → "No"

In [31]:
dataset["high_risk_flag"] = "No"
dataset.loc[(dataset["viewing_level"] == "Low") & (dataset["inactive_over_30_days"] == "Yes"), "high_risk_flag"] = "Yes"

In [32]:
dataset["high_risk_flag"].value_counts()

high_risk_flag
No     4417
Yes     583
Name: count, dtype: int64

### 14. customer_segment
**Purpose:** finds unusual customers who behave against the general pattern.

**Formula:** viewing_level = "High" AND churned = 1 → "Heavy viewer who left"; viewing_level = "Low" AND churned = 0 → "Light viewer who stayed"; everyone else → "Typical"

In [33]:
dataset["customer_segment"] = "Typical"
dataset.loc[(dataset["viewing_level"] == "High") & (dataset["churned"] == 1), "customer_segment"] = "Heavy viewer who left"
dataset.loc[(dataset["viewing_level"] == "Low") & (dataset["churned"] == 0), "customer_segment"] = "Light viewer who stayed"

### 15. watch_time_mismatch
**Purpose:** data quality check showing how many original `avg_watch_time_per_day` values do not match the recalculated values.

**Formula:** difference between avg_watch_time_per_day and avg_watch_recalculated is more than 1 minute in either direction → "Yes", otherwise → "No"

In [34]:
dataset["watch_time_mismatch"] = "No"
dataset.loc[(dataset["avg_watch_time_per_day"] - dataset["avg_watch_recalculated"]) > 1, "watch_time_mismatch"] = "Yes"
dataset.loc[(dataset["avg_watch_time_per_day"] - dataset["avg_watch_recalculated"]) < -1, "watch_time_mismatch"] = "Yes"

In [35]:
dataset["watch_time_mismatch"].value_counts()

watch_time_mismatch
No     3653
Yes    1347
Name: count, dtype: int64

In [37]:
dataset.to_csv("Netflix_calculated_fields.csv", index=False, encoding="utf-8-sig")